<a href="https://colab.research.google.com/github/Alozysr/decoder-llm-from-scratch/blob/main/Dil_Modeli_Denemesi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
project_dir = "/content/drive/MyDrive/mini-llm-tr"
os.makedirs(project_dir, exist_ok=True)

In [ ]:
# 1. Gerekli kütüphaneler
!pip install tiktoken datasets numpy -q


In [ ]:
# 2.Burada veri setini indiriyoruz
from datasets import load_dataset

# Gutenberg'in temizlenmiş İngilizce alt kümesi
dataset = load_dataset("sedthh/gutenberg_english")
print(dataset)
print(dataset["train"][0]["TEXT"][:500])  # ilk örneğin başını gör
print(len(dataset["train"]))

Resolving data files:   0%|          | 0/37 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/29 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['TEXT', 'SOURCE', 'METADATA'],
        num_rows: 48284
    })
})
The United States Bill of Rights.
The Ten Original Amendments to the Constitution of the United States
Passed by Congress September 25, 1789
Ratified December 15, 1791


I
Congress shall make no law respecting an establishment of religion, or

prohibiting the free exercise thereof; or abridging the freedom of speech, or of

the press, or the right of the people peaceably to assemble, and to petition the

Government for a redress of grievances.
II
A well-regulated militia, being ne
48284


In [ ]:
print(len(dataset["train"]))

48284


In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import tiktoken
import numpy as np
import os
print(os.cpu_count())

enc = tiktoken.get_encoding("gpt2")

def process(example):
    ids = enc.encode_ordinary(example["TEXT"])  # bos/eos eklemeden encode
    ids.append(enc.eot_token)  # her hikayenin sonuna end-of-text token'ı ekle
    return {"ids": ids, "len": len(ids)}

dataset["train"] = dataset["train"].select(range(20000))
tokenized = dataset.map(
    process,
    remove_columns=dataset["train"].column_names,
    desc="tokenizing",
    num_proc=2,
    writer_batch_size=550,
)
print(len(dataset["train"]))

2


tokenizing (num_proc=2):   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
# 4. Tüm token'ları tek bir .bin dosyasında birleştir (eğitimde hızlı okuma için)
for split, dset in tokenized.items():
    arr_len = np.sum(dset["len"], dtype=np.uint64)
    filename = os.path.join(project_dir, f"{split}.bin")
    dtype = np.uint16  # GPT-2 vocab size 50257 < 65535, uint16 yeterli
    arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))

    idx = 0
    for example in dset:
        arr[idx : idx + example["len"]] = example["ids"]
        idx += example["len"]
    arr.flush()
    print(f"{filename} yazıldı, {arr_len} token")

/content/drive/MyDrive/mini-llm-tr/train.bin yazıldı, 728534976 token


In [ ]:
import os
os.makedirs("mini-llm-tr", exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/mini-llm-tr/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        # output projection
        self.c_proj = nn.Linear(n_embd, n_embd)
        # regularization
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout
        # causal mask to ensure that attention is only performed to the left in the input sequence
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; self-attend: (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

class Block(nn.Module):
    """ Transformer block """
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd=384, n_head=6, n_layer=6, block_size=256, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, "Sequence uzunluğu block_size'ı aşamaz"

        tok_emb = self.token_emb(idx)  # (B, T, n_embd)
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.pos_emb(pos)  # (T, n_embd)

        x = self.drop(tok_emb + pos_emb)  # token + pozisyon bilgisi birleşiyor

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]  # context'i block_size ile sınırla
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # sadece son token'ın logit'i

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx
    class MLP(nn.Module):
      def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)  # genişlet
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(4 * n_embd, n_embd)  # tekrar daralt
        self.dropout = nn.Dropout(dropout)

      def forward(self, x):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x
    class Block(nn.Module):
      def __init__(self, n_embd, n_head, block_size, dropout=0.1):
          super().__init__()
          self.ln1 = nn.LayerNorm(n_embd)
          self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
          self.ln2 = nn.LayerNorm(n_embd)
          self.mlp = MLP(n_embd, dropout)

      def forward(self, x):
          x = x + self.attn(self.ln1(x))  # residual connection + attention
          x = x + self.mlp(self.ln2(x))   # residual connection + MLP
          return x
    class GPT(nn.Module):
      def __init__(self, vocab_size, n_embd=384, n_head=6, n_layer=6, block_size=256, dropout=0.1):
          super().__init__()
          self.block_size = block_size

          self.token_emb = nn.Embedding(vocab_size, n_embd)
          self.pos_emb = nn.Embedding(block_size, n_embd)
          self.drop = nn.Dropout(dropout)

          self.blocks = nn.ModuleList([
              Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
          ])

          self.ln_f = nn.LayerNorm(n_embd)
          self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

      def forward(self, idx, targets=None):
          B, T = idx.shape
          assert T <= self.block_size, "Sequence uzunluğu block_size'ı aşamaz"

          tok_emb = self.token_emb(idx)  # (B, T, n_embd)
          pos = torch.arange(T, device=idx.device)
          pos_emb = self.pos_emb(pos)  # (T, n_embd)

          x = self.drop(tok_emb + pos_emb)  # token + pozisyon bilgisi birleşiyor

          for block in self.blocks:
              x = block(x)

          x = self.ln_f(x)
          logits = self.lm_head(x)  # (B, T, vocab_size)

          loss = None
          if targets is not None:
              loss = F.cross_entropy(
                  logits.view(-1, logits.size(-1)),
                  targets.view(-1)
              )

          return logits, loss

      @torch.no_grad()
      def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
          for _ in range(max_new_tokens):
              idx_cond = idx[:, -self.block_size:]  # context'i block_size ile sınırla
              logits, _ = self(idx_cond)
              logits = logits[:, -1, :] / temperature  # sadece son token'ın logit'i

              if top_k is not None:
                  v, _ = torch.topk(logits, top_k)
                  logits[logits < v[:, [-1]]] = float('-inf')

              probs = F.softmax(logits, dim=-1)
              idx_next = torch.multinomial(probs, num_samples=1)
              idx = torch.cat((idx, idx_next), dim=1)

          return idx

Overwriting /content/drive/MyDrive/mini-llm-tr/model.py


In [ ]:
import torch
import numpy as np

# ---- Ayarlar ----
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Kullanılan cihaz:", device)

block_size = 128  # context length
batch_size = 12      # her adımda kaç örnek işlenecek
max_iters = 6000      # toplam eğitim adımı
eval_interval = 250    # kaç adımda bir validation loss ölçülecek
eval_iters = 50         # validation loss ölçerken kaç batch ortalanacak
learning_rate = 3e-4

n_embd = 256
n_head = 8
n_layer = 6
dropout = 0.1

Kullanılan cihaz: cuda


In [ ]:
import os
import numpy as np # numpy'nin burada içe aktarıldığından emin olun
import torch # torch'un burada içe aktarıldığından emin olun

# ---- Veriyi yükle (bir önceki adımda oluşturduğumuz .bin dosyaları) ----
# project_dir değişkeni zaten tanımlanmış olmalı (/content/drive/MyDrive/mini-llm-tr)
train_bin_path = os.path.join(project_dir, "train.bin")
val_bin_path = os.path.join(project_dir, "validation.bin")

print(f"train.bin aranıyor: {train_bin_path}")
if not os.path.exists(train_bin_path):
    raise FileNotFoundError(f"HATA: train.bin dosyası bulunamadı: {train_bin_path}. Lütfen Iyv49e7KU82Q hücresinin başarıyla çalıştığından emin olun.")
else:
    print(f"train.bin bulundu: {train_bin_path}")

print(f"validation.bin aranıyor: {val_bin_path}")
if not os.path.exists(val_bin_path):
    raise FileNotFoundError(f"HATA: validation.bin dosyası bulunamadı: {val_bin_path}. Lütfen Iyv49e7KU82Q hücresinin başarıyla çalıştığından emin olun.")
else:
    print(f"validation.bin bulundu: {val_bin_path}")

train_data = np.memmap(train_bin_path, dtype=np.uint16, mode="r")
val_data = np.memmap(val_bin_path, dtype=np.uint16, mode="r")

def get_batch(split):
    data = train_data if split == "train" else val_data
    # rastgele başlangıç noktaları seç
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

train.bin aranıyor: /content/drive/MyDrive/mini-llm-tr/train.bin
train.bin bulundu: /content/drive/MyDrive/mini-llm-tr/train.bin
validation.bin aranıyor: /content/drive/MyDrive/mini-llm-tr/validation.bin
validation.bin bulundu: /content/drive/MyDrive/mini-llm-tr/validation.bin


In [ ]:
# ---- Modeli oluştur ----
import tiktoken
import sys
import os

# model.py dosyasının bulunduğu dizini Python yoluna ekle
project_dir = "/content/drive/MyDrive/mini-llm-tr"
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from model import GPT # GPT sınıfını model.py dosyasından içe aktar

enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab  # 50257

model = GPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)
print(sum(p.numel() for p in model.parameters())/1e6, "M parametre")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

30.503424 M parametre


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {total_params:,}")

Toplam parametre: 30,503,424


In [ ]:
print(f"n_embd: {n_embd}, n_head: {n_head}, n_layer: {n_layer}, block_size: {block_size}")

n_embd: 256, n_head: 8, n_layer: 6, block_size: 128


In [ ]:
# ---- Loss ölçme fonksiyonu (train + val) ----
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split if split == "train" else "val")
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
import time

# Eğitim döngüsü
print("Eğitim başlıyor...")
start_time = time.time()

# ---- Early stopping ayarları ----
patience = 4         # kaç eval_interval boyunca iyileşme olmazsa dur
best_val_loss = float('inf')
patience_counter = 0
best_model_path = "/content/best_model.pt"

#Ana eğitim döngüsü
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"adım {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        if losses['val'] < best_val_loss:
            # Val loss iyileşti — en iyi modeli kaydet, sayaç sıfırla
            best_val_loss = losses['val']
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'iter': iter,
                'val_loss': best_val_loss,
                'config': {
                    'vocab_size': vocab_size,
                    'n_embd': n_embd,
                    'n_head': n_head,
                    'n_layer': n_layer,
                    'block_size': block_size,
                }
            }, best_model_path)
            print(f"  → yeni en iyi model kaydedildi (val loss: {best_val_loss:.4f})")
        else:
            # İyileşme yok — sayaç artır
            patience_counter += 1
            print(f"  → iyileşme yok ({patience_counter}/{patience})")

            if patience_counter >= patience:
                print(f"\nEarly stopping tetiklendi! adım {iter}'de durduruldu.")
                print(f"En iyi val loss: {best_val_loss:.4f}")
                break

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Eğitim tamamlandı!")

Eğitim başlıyor...
adım 0: train loss 11.0167, val loss 10.9526
  → yeni en iyi model kaydedildi (val loss: 10.9526)
adım 250: train loss 4.7541, val loss 7.1765
  → yeni en iyi model kaydedildi (val loss: 7.1765)
adım 500: train loss 4.5456, val loss 6.8449
  → yeni en iyi model kaydedildi (val loss: 6.8449)
adım 750: train loss 4.3783, val loss 6.7329
  → yeni en iyi model kaydedildi (val loss: 6.7329)
adım 1000: train loss 4.3052, val loss 6.6873
  → yeni en iyi model kaydedildi (val loss: 6.6873)
adım 1250: train loss 4.2540, val loss 6.5419
  → yeni en iyi model kaydedildi (val loss: 6.5419)
adım 1500: train loss 4.1284, val loss 6.4149
  → yeni en iyi model kaydedildi (val loss: 6.4149)
adım 1750: train loss 4.0740, val loss 6.4138
  → yeni en iyi model kaydedildi (val loss: 6.4138)
adım 2000: train loss 4.0002, val loss 6.3302
  → yeni en iyi model kaydedildi (val loss: 6.3302)
adım 2250: train loss 4.0027, val loss 6.4179
  → iyileşme yok (1/4)
adım 2500: train loss 3.9096, val

### 7. Eğitilmiş Modeli Kaydetme
Eğitim tamamlandıktan sonra, eğitilmiş modelin son halini kaydedelim.

In [ ]:
# Eğitilmiş modelin son halini kaydet
final_model_path = os.path.join(project_dir, "final_model.pt")
torch.save(model.state_dict(), final_model_path)
print(f"Final model kaydedildi: {final_model_path}")

Final model kaydedildi: /content/drive/MyDrive/mini-llm-tr/final_model.pt


### 8. Metin Oluşturma (Test)
Eğitilmiş modelimizi kullanarak metin oluşturalım.

In [ ]:
# Kaydedilmiş modeli yükle
model.load_state_dict(torch.load(final_model_path))
model.eval() # Değerlendirme moduna al

# Başlangıç metni (prompt)
start_text = "The man"

# Metni token'lara dönüştür
context = torch.tensor(enc.encode(start_text), dtype=torch.long, device=device).unsqueeze(0)

# Metin oluştur
generated_tokens = model.generate(context, max_new_tokens=175)

# Token'ları tekrar metne dönüştür
generated_text = enc.decode(generated_tokens[0].tolist())
print("Oluşturulan metin:")
print(generated_text)

Oluşturulan metin:
The man Erently,

      and in having arisen on may not be boarding to us by any difficulty

      knew how their face were russianiness with his rifle, that did not

      try him unto him, that, without being only given of thine with the

      regions upon it, and the hearts of the best of his education.

    


      “No!” saith the armfully was interpreted sharp in the

      college. “Did first find things with ordinary effect, by me, 163,

      to a lady in that!�
